# Hospital Analytics — Excel to MySQL
यह notebook पाँच Excel workbooks की हर sheet को `hospital_analytics` नाम के MySQL database में अलग table के रूप में import करती है।

**ज़रूरी:** यह notebook local Jupyter/Anaconda/VS Code में चलाएँ; MySQL Server installed और running होना चाहिए। पाँचों Excel files notebook के उसी folder में रखें। Import code same-named tables को replace करता है—पुराना data हो तो पहले backup लें।


## 1. Libraries install करें

In [ ]:
%pip install pandas openpyxl sqlalchemy mysql-connector-python

## 2. Excel files और sheets verify करें

In [ ]:
from pathlib import Path
import pandas as pd
import re

FILES = {
    "1_Hospital_Operation.xlsx": "hospital",
    "2_Patient_Analytics.xlsx": "patient",
    "3_Disease_Analytics.xlsx": "disease",
    "4_Doctor_Performance.xlsx": "doctor",
    "5_Cost_and_Revenue.xlsx": "cost_revenue",
}

missing = [f for f in FILES if not Path(f).exists()]
if missing:
    raise FileNotFoundError(
        "Files notebook folder mein nahi milin: " + ", ".join(missing)
        + ". Excel files ko notebook ke same folder mein rakhein."
    )

for filename in FILES:
    print("\\n" + filename)
    print(pd.ExcelFile(filename).sheet_names)


## 3. MySQL database create और connect करें

In [ ]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

MYSQL_HOST = input("MySQL host [localhost]: ").strip() or "localhost"
MYSQL_PORT = int(input("MySQL port [3306]: ").strip() or "3306")
MYSQL_USER = input("MySQL username [root]: ").strip() or "root"
MYSQL_PASSWORD = getpass("MySQL password: ")

DATABASE_NAME = "hospital_analytics"

server_url = URL.create(
    "mysql+mysqlconnector",
    username=MYSQL_USER, password=MYSQL_PASSWORD,
    host=MYSQL_HOST, port=MYSQL_PORT
)
server_engine = create_engine(server_url, pool_pre_ping=True)

with server_engine.connect() as conn:
    conn.execute(text(
        f"CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}` "
        "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
    ))
    conn.commit()

db_url = URL.create(
    "mysql+mysqlconnector",
    username=MYSQL_USER, password=MYSQL_PASSWORD,
    host=MYSQL_HOST, port=MYSQL_PORT, database=DATABASE_NAME
)
engine = create_engine(db_url, pool_pre_ping=True)
with engine.connect():
    print("Connected successfully to", DATABASE_NAME)


## 4. हर workbook की हर sheet को MySQL table में import करें

In [ ]:
def clean_sql_name(value):
    name = re.sub(r"[^a-zA-Z0-9]+", "_", str(value).strip().lower()).strip("_")
    if not name:
        name = "unnamed"
    if name[0].isdigit():
        name = "t_" + name
    return name

import_results = []

for filename, prefix in FILES.items():
    workbook = pd.ExcelFile(filename)
    for sheet_name in workbook.sheet_names:
        df = pd.read_excel(filename, sheet_name=sheet_name)
        df.columns = [clean_sql_name(c) for c in df.columns]

        # Cleaned column names unique बनाएं
        seen = {}
        unique_cols = []
        for col in df.columns:
            seen[col] = seen.get(col, 0) + 1
            unique_cols.append(col if seen[col] == 1 else f"{col}_{seen[col]}")
        df.columns = unique_cols

        table_name = f"{prefix}_{clean_sql_name(sheet_name)}"
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",
            index=False,
            chunksize=1000
        )
        import_results.append({
            "source_file": filename,
            "source_sheet": sheet_name,
            "mysql_table": table_name,
            "rows_imported": len(df),
            "columns_imported": len(df.columns)
        })
        print(f"Imported {table_name}: {len(df)} rows")

results_df = pd.DataFrame(import_results)
display(results_df)
print("Total tables imported:", len(results_df))


## 5. Tables verify करें

In [ ]:
with engine.connect() as conn:
    tables_df = pd.read_sql(
        text("SELECT table_name FROM information_schema.tables "
             "WHERE table_schema = :db ORDER BY table_name"),
        conn,
        params={"db": DATABASE_NAME}
    )
display(tables_df)
print("Tables in database:", len(tables_df))


## 6. Sample table का data देखें

In [ ]:
# Example table; table name imported above
sample_table = "hospital_raw_data"
with engine.connect() as conn:
    sample_df = pd.read_sql(
        text(f"SELECT * FROM `{sample_table}` LIMIT 10"), conn
    )
display(sample_df)


## Common errors
- `Can't connect to MySQL server`: MySQL Server start करें और host/port जाँचें।
- `Access denied`: username/password और CREATE DATABASE permission जाँचें।
- `File not found`: पाँचों Excel files notebook के same folder में रखें।
- `ModuleNotFoundError`: पहला cell चलाएँ और जरूरत हो तो kernel restart करें।
- JupyterLite/browser-only notebook आम तौर पर local MySQL Server से connect नहीं कर सकती; local Jupyter/Anaconda या network-accessible server इस्तेमाल करें।

यह notebook tables import करती है, लेकिन foreign keys/relationships अपने-आप नहीं बनाती। Relationship बनाने से पहले common IDs verify करें।
